# Data Warehouse & OLAP Cube Simulation
This notebook demonstrates building a Star Schema Data Warehouse and performing an OLAP cube aggregation.

In [ ]:
import sqlite3
import pandas as pd

# Connect to an in-memory database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("--- BUILDING THE DATA WAREHOUSE (STAR SCHEMA) ---")

# Create Dimension Tables
cursor.execute('''
CREATE TABLE dim_product (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT
)
''')

cursor.execute('''
CREATE TABLE dim_store (
    store_id INTEGER PRIMARY KEY,
    store_name TEXT,
    region TEXT
)
''')

cursor.execute('''
CREATE TABLE dim_time (
    time_id INTEGER PRIMARY KEY,
    date TEXT,
    month TEXT,
    year INTEGER
)
''')

# Create Fact Table
cursor.execute('''
CREATE TABLE fact_sales (
    sale_id INTEGER PRIMARY KEY,
    product_id INTEGER,
    store_id INTEGER,
    time_id INTEGER,
    quantity INTEGER,
    revenue REAL,
    FOREIGN KEY(product_id) REFERENCES dim_product(product_id),
    FOREIGN KEY(store_id) REFERENCES dim_store(store_id),
    FOREIGN KEY(time_id) REFERENCES dim_time(time_id)
)
''')

# Populate data
cursor.executemany("INSERT INTO dim_product VALUES (?, ?, ?)", [(1, 'Laptop', 'Electronics'), (2, 'Desk', 'Furniture'), (3, 'Chair', 'Furniture')])
cursor.executemany("INSERT INTO dim_store VALUES (?, ?, ?)", [(1, 'Downtown Store', 'East'), (2, 'Uptown Store', 'West')])
cursor.executemany("INSERT INTO dim_time VALUES (?, ?, ?, ?)", [(1, '2023-01-15', 'January', 2023), (2, '2023-01-20', 'January', 2023), (3, '2023-02-10', 'February', 2023)])
cursor.executemany("INSERT INTO fact_sales VALUES (?, ?, ?, ?, ?, ?)", [(1, 1, 1, 1, 2, 2000.0), (2, 2, 1, 1, 1, 500.0), (3, 3, 2, 2, 4, 600.0), (4, 1, 2, 3, 1, 1000.0)])
conn.commit()
print("Warehouse populated successfully!")

In [ ]:
print("\n--- LISTING THE DIMENSIONS ---")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
dimensions = [t[0] for t in cursor.fetchall() if t[0].startswith('dim_')]
for dim in dimensions:
    print(f" - {dim}")

In [ ]:
print("\n--- SIMULATING OLAP CUBE QUERIES ---")
query = '''
SELECT s.region, p.category, SUM(f.revenue) as total_revenue
FROM fact_sales f
JOIN dim_store s ON f.store_id = s.store_id
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY s.region, p.category
ORDER BY s.region, p.category;
'''
df_cube = pd.read_sql_query(query, conn)
display(df_cube)